# Convert PSD Results from Code Units to Physical (CGS/Gaussian) Units

Loads the truncated PSD output from notebook 07 (hybrid-code normalized units)
and converts to CGS (Gaussian) units for a specified magnetic field strength B₀
and number density n₀. Code normalization: B→B₀, length→dᵢ, time→Ωci⁻¹,
velocity→V_A.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import os

%matplotlib inline

# CGS (Gaussian) constants
m_p = 1.67262192e-24   # proton mass [g]
e_esu = 4.80320451e-10  # elementary charge [statC / esu]
c_cgs = 2.99792458e10   # speed of light [cm/s]

In [ ]:
# ============================================================
# User-specified physical parameters — edit these (CGS units)
# ============================================================
B0 = 0.000659     # G  (= 0.01 T, typical coronal loop)
n0 = 1e9        # cm^-3  (= 10^15 m^-3, typical coronal loop)

#TRUNC_FILE = "lagrangian_batch_truncated.nc"
#SAVE_NAME  = "lagrangian_batch_truncated_cgs.nc"
TRUNC_FILE = "/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/thickCS_data/thicker_lagrangian_batch.nc"
SAVE_NAME  = "/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/thickCS_data/processed_thicker_lagrangian_batch.nc"
PSD_DIR    = os.path.join(os.path.dirname(os.getcwd()), "psd_plots_cgs") if os.path.basename(os.getcwd()) == "notebooks" else "psd_plots_cgs"
os.makedirs(PSD_DIR, exist_ok=True)

## Derive Physical Scales

In [ ]:
# Fundamental scales from B0 and n0 (CGS/Gaussian)
Omega_ci = e_esu * B0 / (m_p * c_cgs)                    # ion cyclotron frequency [rad/s]
f_ci     = Omega_ci / (2 * np.pi)                         # ion cyclotron frequency [Hz]
omega_pi = np.sqrt(4 * np.pi * n0 * e_esu**2 / m_p)      # ion plasma frequency [rad/s]
d_i      = c_cgs / omega_pi                               # ion inertial length [cm]
V_A      = B0 / np.sqrt(4 * np.pi * n0 * m_p)            # Alfven speed [cm/s]

print("Derived physical scales (CGS)")
print("=" * 50)
print(f"  Omega_ci  = {Omega_ci:.4e} rad/s  ({f_ci:.4e} Hz)")
print(f"  omega_pi  = {omega_pi:.4e} rad/s")
print(f"  d_i       = {d_i:.4e} cm  ({d_i/1e5:.4f} km)")
print(f"  V_A       = {V_A:.4e} cm/s  ({V_A/1e5:.2f} km/s)")
print(f"  V_A / c   = {V_A/c_cgs:.4e}")
print(f"  B0^2/4pi  = {B0**2/(4*np.pi):.4e} erg/cm^3")
print(f"  n0*m_p*V_A^2 = {n0*m_p*V_A**2:.4e} erg/cm^3  (should equal B0^2/4pi)")

## Define Conversion Factors (CGS/Gaussian)

In Gaussian CGS the E and B fields have the same dimensions (statV/cm = G).
The motional electric field is **E = -(v/c) x B**, so the code-unit E field
maps to (V_A/c) B₀.

| Quantity | Code unit | CGS scale factor |
|----------|-----------|-------------------|
| frequency | Ωci | f_Hz = f_code x Ωci |
| B field | B₀ | B_CGS = B_code x B₀  [G] |
| E field | (V_A/c) B₀ | E_CGS = E_code x (V_A/c) B₀  [statV/cm] |
| Poynting (c/4pi)(ExB) | n₀ m_p V_A³ | S_CGS = S_code x n₀ m_p V_A³  [erg/cm²/s] |
| Energy density | n₀ m_p V_A² = B₀²/4pi | u_CGS = u_code x B₀²/4pi  [erg/cm³] |

For PSDs: PSD_CGS = PSD_code x Q² / Ωci  (where Q is the field scale factor).

In [ ]:
# Field scale factors (code unit -> CGS)
B_scale = B0                            # G
E_scale = (V_A / c_cgs) * B0           # statV/cm  (E = v/c x B in Gaussian)
S_scale = n0 * m_p * V_A**3            # erg/cm^2/s  (= c/4pi * E_scale * B_scale)
u_scale = B0**2 / (4 * np.pi)          # erg/cm^3  (= n0 * m_p * V_A^2)
freq_scale = Omega_ci                   # rad/s -> Hz conversion factor

# PSD conversion: PSD_CGS = PSD_code * Q^2 / freq_scale
# Maps component name -> (psd_scale_factor, y-axis label)
CONV = {
    "Ex": (E_scale**2 / freq_scale, r"PSD [(statV/cm)$^2$/Hz]"),
    "Ey": (E_scale**2 / freq_scale, r"PSD [(statV/cm)$^2$/Hz]"),
    "Ez": (E_scale**2 / freq_scale, r"PSD [(statV/cm)$^2$/Hz]"),
    "Bx": (B_scale**2 / freq_scale, r"PSD [G$^2$/Hz]"),
    "By": (B_scale**2 / freq_scale, r"PSD [G$^2$/Hz]"),
    "Bz": (B_scale**2 / freq_scale, r"PSD [G$^2$/Hz]"),
    "Sx": (S_scale**2 / freq_scale, r"PSD [(erg/cm$^2$/s)$^2$/Hz]"),
    "Sy": (S_scale**2 / freq_scale, r"PSD [(erg/cm$^2$/s)$^2$/Hz]"),
    "Sz": (S_scale**2 / freq_scale, r"PSD [(erg/cm$^2$/s)$^2$/Hz]"),
    "uE": (u_scale**2 / freq_scale, r"PSD [(erg/cm$^3$)$^2$/Hz]"),
    "uB": (u_scale**2 / freq_scale, r"PSD [(erg/cm$^3$)$^2$/Hz]"),
}

print("PSD conversion factors (PSD_CGS = factor * PSD_code):")
print("-" * 60)
for comp, (scale, label) in CONV.items():
    print(f"  {comp:4s}: {scale:.4e}   {label}")

## Load Data & Convert

In [ ]:
ds = xr.open_dataset(TRUNC_FILE)
print(ds)
print(f"\ndt (code) = {ds.attrs['dt']}")

# Convert frequency coordinate
freq_code = ds["frequency"].values
freq_hz   = freq_code * freq_scale  # Hz

# Convert spatial coordinates
x0_cgs = ds["x0"].values * d_i   # cm
y0_cgs = ds["y0"].values * d_i   # cm

# Detect which PSD components are available
ALL_COMPS_FULL = ["Ex", "Ey", "Ez", "Bx", "By", "Bz", "Sx", "Sy", "Sz", "uE", "uB"]
ALL_COMPS = [c for c in ALL_COMPS_FULL if f"psd_{c}" in ds]
print(f"\nAvailable PSD components: {ALL_COMPS}")

# Handle optional fields (absent when file comes from notebook 06 without truncation)
N_total = len(ds["x0"])
N_freq = len(freq_code)

if "end_time_idx" in ds:
    end_time_idx = ds["end_time_idx"].values
else:
    # No truncation — all trajectories use the full frequency array
    end_time_idx = np.full(N_total, N_freq, dtype=int)
    print("(end_time_idx not found — assuming full trajectories)")

if "n_freq" in ds:
    n_freq = ds["n_freq"].values
else:
    # Every trajectory has the same frequency grid
    n_freq = np.full(N_total, N_freq, dtype=int)
    print("(n_freq not found — assuming full frequency grid for all trajectories)")

# Build CGS dataset
data_vars = {
    "end_time_idx": (["trajectory"], end_time_idx),
    "x0_code":      (["trajectory"], ds["x0"].values),
    "y0_code":      (["trajectory"], ds["y0"].values),
    "x0":           (["trajectory"], x0_cgs),
    "y0":           (["trajectory"], y0_cgs),
    "n_freq":       (["trajectory"], n_freq),
}

for comp in ALL_COMPS:
    scale, _ = CONV[comp]
    data_vars[f"psd_{comp}"] = (
        ["trajectory", "frequency"],
        ds[f"psd_{comp}"].values * scale,
    )

ds_cgs = xr.Dataset(
    data_vars,
    coords={
        "trajectory": ds["trajectory"].values,
        "frequency":  freq_hz,
    },
    attrs={
        "source_file":  ds.attrs.get("source_file", ""),
        "psd_method":   ds.attrs.get("psd_method", ""),
        "dt_code":      ds.attrs["dt"],
        "dt_cgs":       ds.attrs["dt"] / Omega_ci,
        "B0_G":         B0,
        "n0_cm3":       n0,
        "Omega_ci":     Omega_ci,
        "f_ci_Hz":      f_ci,
        "d_i_cm":       d_i,
        "V_A_cms":      V_A,
        "freq_unit":    "Hz",
        "x0_unit":      "cm",
        "y0_unit":      "cm",
        "unit_system":  "CGS (Gaussian)",
    },
)

ds.close()
print("\nConverted dataset:")
print(ds_cgs)

## Sanity Checks

In [ ]:
print("Sanity checks")
print("=" * 60)

# Frequency range
f_max = freq_hz[~np.isnan(freq_hz)].max()
print(f"Frequency range: 0 to {f_max:.2e} Hz  ({f_max/1e3:.2f} kHz)")
print(f"  f_ci = {f_ci:.2e} Hz  ({f_ci/1e3:.2f} kHz)")
print(f"  f_max / f_ci = {f_max / f_ci:.2f}")

# Peak PSD values for a representative trajectory
traj_idx = 0
n_f = int(ds_cgs["n_freq"].values[traj_idx])
print(f"\nTrajectory {traj_idx} (n_freq={n_f}):")
for comp in ALL_COMPS:
    psd_vals = ds_cgs[f"psd_{comp}"].values[traj_idx, 1:n_f]  # skip DC
    valid = psd_vals[~np.isnan(psd_vals) & (psd_vals > 0)]
    if len(valid) > 0:
        ipeak = np.argmax(valid)
        f_peak = freq_hz[1:n_f][ipeak]
        _, label = CONV[comp]
        print(f"  {comp:4s}: peak = {valid[ipeak]:.4e}  at f = {f_peak:.4e} Hz  ({f_peak/1e3:.2f} kHz)")

# Parseval check: integral of PSD ~ variance
if "Bz" in ALL_COMPS:
    print(f"\nParseval check (trajectory {traj_idx}, Bz):")
    psd_bz = ds_cgs["psd_Bz"].values[traj_idx, :n_f]
    f_arr  = freq_hz[:n_f]
    df = f_arr[1] - f_arr[0]
    integral = np.nansum(psd_bz) * df
    print(f"  integral(PSD_Bz) * df = {integral:.4e} G^2")
    print(f"  sqrt(integral)        = {np.sqrt(integral):.4e} G")
    print(f"  (For reference, B0 = {B0:.4e} G)")

## Plot Converted PSDs

In [ ]:
E_COMPS = [c for c in ["Ex", "Ey", "Ez"] if c in ALL_COMPS]
B_COMPS = [c for c in ["Bx", "By", "Bz"] if c in ALL_COMPS]
S_COMPS = [c for c in ["Sx", "Sy", "Sz"] if c in ALL_COMPS]
ENERGY_COMPS = [c for c in ["uE", "uB"] if c in ALL_COMPS]
COMP_COLORS = {
    "Ex": "C0", "Ey": "C1", "Ez": "C2",
    "Bx": "C3", "By": "C4", "Bz": "C5",
    "Sx": "C0", "Sy": "C1", "Sz": "C2",
    "uE": "C6", "uB": "C7",
}

N_total = len(ds_cgs["trajectory"])
x0s_code = ds_cgs["x0_code"].values
y0s_code = ds_cgs["y0_code"].values

# --- Compute global axis limits ---
all_freqs_min, all_freqs_max = np.inf, -np.inf
e_psd_min, e_psd_max = np.inf, -np.inf
b_psd_min, b_psd_max = np.inf, -np.inf
d_psd_min, d_psd_max = np.inf, -np.inf

for traj_idx in range(N_total):
    n_f = int(ds_cgs["n_freq"].values[traj_idx])
    f = freq_hz[1:n_f]
    if len(f) == 0:
        continue
    all_freqs_min = min(all_freqs_min, f[0])
    all_freqs_max = max(all_freqs_max, f[-1])
    for comp in ALL_COMPS:
        p = ds_cgs[f"psd_{comp}"].values[traj_idx, 1:n_f]
        p_valid = p[(~np.isnan(p)) & (p > 0)]
        if len(p_valid) == 0:
            continue
        if comp in E_COMPS:
            e_psd_min = min(e_psd_min, p_valid.min())
            e_psd_max = max(e_psd_max, p_valid.max())
        elif comp in B_COMPS:
            b_psd_min = min(b_psd_min, p_valid.min())
            b_psd_max = max(b_psd_max, p_valid.max())
        else:
            d_psd_min = min(d_psd_min, p_valid.min())
            d_psd_max = max(d_psd_max, p_valid.max())

FREQ_LIM  = (all_freqs_min * 0.8, all_freqs_max * 1.2)
E_PSD_LIM = (e_psd_min * 0.3, e_psd_max * 3.0) if E_COMPS else (1, 1)
B_PSD_LIM = (b_psd_min * 0.3, b_psd_max * 3.0) if B_COMPS else (1, 1)
D_PSD_LIM = (d_psd_min * 0.3, d_psd_max * 3.0) if (S_COMPS or ENERGY_COMPS) else (1, 1)

print(f"Freq limits [Hz]:      {FREQ_LIM[0]:.2e} -- {FREQ_LIM[1]:.2e}")
if E_COMPS:
    print(f"E PSD limits:          {E_PSD_LIM[0]:.2e} -- {E_PSD_LIM[1]:.2e}")
if B_COMPS:
    print(f"B PSD limits:          {B_PSD_LIM[0]:.2e} -- {B_PSD_LIM[1]:.2e}")
if S_COMPS or ENERGY_COMPS:
    print(f"Derived PSD limits:    {D_PSD_LIM[0]:.2e} -- {D_PSD_LIM[1]:.2e}")

In [ ]:
# Build list of panels to plot
panels = []
if E_COMPS:
    panels.append(("E-field", E_COMPS, E_PSD_LIM, CONV["Ex"][1]))
if B_COMPS:
    panels.append(("B-field", B_COMPS, B_PSD_LIM, CONV["Bx"][1]))
if S_COMPS or ENERGY_COMPS:
    panels.append(("Poynting & Energy", S_COMPS + ENERGY_COMPS, D_PSD_LIM, CONV.get("Sx", CONV.get("uE", (None, "PSD")))[1]))
n_panels = len(panels)

for traj_idx in range(N_total):
    n_f = int(ds_cgs["n_freq"].values[traj_idx])
    x0 = x0s_code[traj_idx]
    y0 = y0s_code[traj_idx]
    end = int(ds_cgs["end_time_idx"].values[traj_idx])

    f = freq_hz[:n_f]

    fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 5))
    if n_panels == 1:
        axes = [axes]

    for ax, (title, comps, ylim, ylabel) in zip(axes, panels):
        for comp in comps:
            p = ds_cgs[f"psd_{comp}"].values[traj_idx, :n_f]
            ls = "--" if comp in ENERGY_COMPS else "-"
            ax.loglog(f[1:], p[1:], color=COMP_COLORS[comp], lw=1.2,
                      linestyle=ls, label=comp)
        ax.set_xlim(FREQ_LIM)
        ax.set_ylim(ylim)
        ax.set_xlabel("frequency [Hz]")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, which="both")

    fig.suptitle(
        f"Trajectory {traj_idx:02d}  "
        f"(x0={x0:.1f} $d_i$, y0={y0:.1f} $d_i$)  "
        f"{end} steps  [CGS units]",
        fontsize=12,
    )
    plt.tight_layout()

    fname = os.path.join(
        PSD_DIR,
        f"psd_cgs_{traj_idx:02d}_x0_{x0:.1f}_y0_{y0:.1f}.png",
    )
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Saved {N_total} CGS PSD plots to {PSD_DIR}/")

## Save Converted Dataset

In [ ]:
out_path = Path(".") / SAVE_NAME
ds_cgs.to_netcdf(out_path)
print(f"Saved to {out_path}")
print(ds_cgs)

In [ ]:
# I want to play around with the data a little bit to make sure that this is all making sense
ds = xr.open_dataset(TRUNC_FILE)
print(ds)
print(f"\ndt (code) = {ds.attrs['dt']}")

# Convert frequency coordinate
freq_code = ds["frequency"].values
#freq_hz   = freq_code * freq_scale  # Hz

# Convert spatial coordinates
x0_code = ds["x0"].values
y0_code = ds["y0"].values

# Detect which PSD components are available
ALL_COMPS_FULL = ["Ex", "Ey", "Ez", "Bx", "By", "Bz", "Sx", "Sy", "Sz", "uE", "uB"]
ALL_COMPS = [c for c in ALL_COMPS_FULL if f"psd_{c}" in ds]
print(f"\nAvailable PSD components: {ALL_COMPS}")

print(ds['Bx_sampled'][4].values)


In [ ]:
wpt = 4

tt = ds['time'].values

bx = ds['Bx_sampled'][wpt].values
by = ds['By_sampled'][wpt].values
bz = ds['Bz_sampled'][wpt].values
bb = np.sqrt(bx**2 + by**2 + bz**2)

ex = ds['Ex_sampled'][wpt].values
ey = ds['Ey_sampled'][wpt].values
ez = ds['Ez_sampled'][wpt].values
ee = np.sqrt(ex**2 + ey**2 + ez**2)

edb = bx*ex + by*ey + bz*ez

plt.clf()
plt.subplot(211)
plt.plot(tt, bx)
plt.plot(tt, by)
plt.plot(tt, bz)
plt.plot(tt, bb)

plt.subplot(212)
# plt.plot(tt, ex)
# plt.plot(tt, ey)
# plt.plot(tt, ez)
# plt.plot(tt, ee)
plt.plot(tt, edb)
